# 04 — Support Vector Regression (lineal vs RBF)

**TP Final · Aprendizaje de Máquina I (CEIA-FIUBA)** · Jaime Pinzón (a2629)

## Justificación del modelo (criterio oficial 5)

**Qué es.** SVR busca una función que deje a la mayor cantidad de observaciones dentro de un **tubo de tolerancia ε**: los errores menores a ε no se penalizan (pérdida ε-insensible) y el hiperparámetro **C** regula cuánto se penalizan los que quedan afuera. La solución queda determinada solo por los **vectores de soporte** (los puntos en el borde o fuera del tubo). Con la variante kernel (RBF) la frontera se vuelve no lineal proyectando implícitamente a un espacio de mayor dimensión.

**Por qué tiene sentido acá.** (i) La pérdida ε-insensible es conceptualmente afín a nuestra métrica de negocio: errores chicos en días "no importan" y el MAE castiga linealmente los grandes — a diferencia de OLS que castiga cuadráticamente y se deja arrastrar por la cola alta. (ii) El contraste lineal-vs-RBF separa cuánta ganancia viene de la **no linealidad global** (RBF) frente a la local de KNN. (iii) Es el segundo modelo de la materia (clase 3) y completa el mapa paramétrico↔no paramétrico.

**Qué exige de los datos.** Escalado obligatorio (`build_preprocessor(scale=True)`): el kernel RBF es una función de distancias euclídeas (mismo argumento que KNN — `num_lab_procedures` 1–94 aplastaría al resto), y en el caso lineal C y ε cambian de significado con la escala de las features.

**Ventajas / desventajas.**
- ✅ Robusto por la pérdida ε-insensible; frontera global suave; kernels dan no linealidad sin ingeniería de features.
- ❌ Costo de entrenamiento O(n²)–O(n³) con kernel: es el modelo más caro del TP. Estrategia (R2 del plan): LinearSVR (implementación lineal, escala bien) sobre el train completo; para RBF, **búsqueda** de hiperparámetros sobre una submuestra estratificada del train y **refit final sobre el train completo** (la sonda de tiempos mostró que un único fit completo es viable: ~2–4 min).

**Hiperparámetros:** LinearSVR: grilla C×ε. SVR-RBF: Optuna (TPE, seed 42, storage sqlite) sobre C, ε y γ — espacio continuo de 3 dimensiones donde la búsqueda bayesiana rinde más que una grilla (D7).

In [1]:
import time

import pandas as pd
from sklearn.model_selection import KFold

from src.config import DIR_PROCESSED, SEED, TARGET
from src.evaluacion import evaluar, registrar
from src.pipelines import build_preprocessor

X_train = pd.read_parquet(DIR_PROCESSED / "X_train.parquet")
X_test = pd.read_parquet(DIR_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(DIR_PROCESSED / "y_train.parquet")[TARGET]
y_test = pd.read_parquet(DIR_PROCESSED / "y_test.parquet")[TARGET]

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"train: {X_train.shape} | test: {X_test.shape}")

train: (53756, 8) | test: (20354, 8)


In [2]:
# --- Variante 1: LinearSVR sobre el train COMPLETO ---
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVR

pipe_lineal = Pipeline([
    ("preprocesador", build_preprocessor(scale=True)),
    ("modelo", LinearSVR(max_iter=20000, random_state=SEED)),
])
grilla_lineal = {
    "modelo__C": [0.01, 0.1, 1.0, 10.0],
    "modelo__epsilon": [0.1, 0.5, 1.0],
}
busqueda_lineal = GridSearchCV(
    pipe_lineal, grilla_lineal, cv=cv,
    scoring="neg_mean_absolute_error", n_jobs=-1, refit=True,
)
t0 = time.perf_counter()
busqueda_lineal.fit(X_train, y_train)
t_lineal = time.perf_counter() - t0

print(f"LinearSVR: 12 configs x 5 folds en {t_lineal/60:.1f} min")
print(f"Mejor config: {busqueda_lineal.best_params_}")
print(f"MAE (CV): {-busqueda_lineal.best_score_:.4f} dias")

metricas_lineal = evaluar(busqueda_lineal.best_estimator_, X_test, y_test)
metricas_lineal["tiempo_s"] = round(busqueda_lineal.refit_time_, 2)
registrar(
    "svr_lineal", metricas_lineal,
    notas=(f"LinearSVR C={busqueda_lineal.best_params_['modelo__C']}, "
           f"eps={busqueda_lineal.best_params_['modelo__epsilon']}; "
           f"train completo; busqueda {t_lineal/60:.1f} min"),
)
print({k: round(v, 4) for k, v in metricas_lineal.items()})

LinearSVR: 12 configs x 5 folds en 6.9 min
Mejor config: {'modelo__C': 0.01, 'modelo__epsilon': 0.5}
MAE (CV): 1.6795 dias
{'mae': 1.8343, 'rmse': 2.5168, 'r2': 0.2891, 'tiempo_s': 0.3}


In [3]:
# --- Submuestra estratificada para la busqueda RBF: SOLO del train ---
from sklearn.model_selection import train_test_split

X_sub, _, y_sub, _ = train_test_split(
    X_train, y_train, train_size=16000, stratify=y_train, random_state=SEED
)

# Evidencia de que la submuestra proviene exclusivamente del train:
assert X_sub.index.isin(X_train.index).all()
assert not X_sub.index.isin(X_test.index.difference(X_train.index)).any()
print(f"Submuestra: {X_sub.shape[0]:,} filas, todas con indice dentro de X_train "
      f"(rango de indices 0..{len(X_train)-1}, max usado: {X_sub.index.max()})")
print("Estratificacion (proporciones por dias, submuestra vs train):")
comp = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "submuestra": y_sub.value_counts(normalize=True).sort_index(),
}).round(4)
print(comp.T.to_string())

Submuestra: 16,000 filas, todas con indice dentro de X_train (rango de indices 0..53755, max usado: 53753)
Estratificacion (proporciones por dias, submuestra vs train):
time_in_hospital    1.0     2.0     3.0     4.0     5.0     6.0     7.0     8.0     9.0     10.0    11.0    12.0
train             0.1461  0.1784  0.1829  0.1416  0.1004  0.0748  0.0569  0.0417  0.0271  0.0209  0.0165  0.0126
submuestra        0.1461  0.1784  0.1829  0.1416  0.1004  0.0748  0.0569  0.0417  0.0271  0.0209  0.0165  0.0126


In [4]:
# --- Variante 2: SVR-RBF — busqueda Optuna sobre la submuestra ---
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVR

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    params = {
        "C": trial.suggest_float("C", 0.1, 100.0, log=True),
        "epsilon": trial.suggest_float("epsilon", 0.05, 2.0, log=True),
        "gamma": trial.suggest_float("gamma", 1e-3, 1.0, log=True),
    }
    p = Pipeline([
        ("preprocesador", build_preprocessor(scale=True)),
        ("modelo", SVR(kernel="rbf", cache_size=500, **params)),
    ])
    return cross_val_score(
        p, X_sub, y_sub, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1
    ).mean()

estudio = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    storage="sqlite:///optuna.db", study_name="svr_rbf_sub16k",
    load_if_exists=True,
)
t0 = time.perf_counter()
# Presupuesto duro del plan: 2 h (timeout=7200); sqlite permite retomar si se corta
estudio.optimize(objetivo, n_trials=40, timeout=7200)
t_optuna = time.perf_counter() - t0

print(f"Optuna: {len(estudio.trials)} trials en {t_optuna/60:.1f} min")
print(f"Mejor MAE (CV en submuestra): {-estudio.best_value:.4f} dias")
print(f"Mejores hiperparametros: {estudio.best_params}")

Optuna: 40 trials en 18.5 min
Mejor MAE (CV en submuestra): 1.6421 dias
Mejores hiperparametros: {'C': 8.59772614854437, 'epsilon': 0.43966734537697194, 'gamma': 0.009359914130240485}


In [5]:
# Refit final de la mejor config RBF sobre el TRAIN COMPLETO + test (una vez)
pipe_rbf = Pipeline([
    ("preprocesador", build_preprocessor(scale=True)),
    ("modelo", SVR(kernel="rbf", cache_size=500, **estudio.best_params)),
])
t0 = time.perf_counter()
pipe_rbf.fit(X_train, y_train)
t_refit = time.perf_counter() - t0
print(f"Refit sobre train completo (53.756 filas): {t_refit/60:.1f} min")

t0 = time.perf_counter()
metricas_rbf = evaluar(pipe_rbf, X_test, y_test)
t_pred = time.perf_counter() - t0
metricas_rbf["tiempo_s"] = round(t_refit, 2)
registrar(
    "svr_rbf", metricas_rbf,
    notas=(f"SVR-RBF {['%s=%.4g' % (k, v) for k, v in estudio.best_params.items()]}; "
           f"busqueda TPE {len(estudio.trials)} trials en submuestra 16k del train "
           f"({t_optuna/60:.0f} min); refit en train completo ({t_refit/60:.1f} min)"),
)
print({k: round(v, 4) for k, v in metricas_rbf.items()})
print(f"(prediccion sobre 20.354 filas de test: {t_pred:.1f} s)")

Refit sobre train completo (53.756 filas): 1.8 min


{'mae': 1.7687, 'rmse': 2.4425, 'r2': 0.3304, 'tiempo_s': 110.2}
(prediccion sobre 20.354 filas de test: 96.9 s)


## Cierre: lineal vs RBF y el costo de cada día recortado

La comparación de las dos variantes responde una pregunta concreta: **¿cuánta ganancia aporta la no linealidad global, y a qué costo computacional?** LinearSVR entrena en segundos; el RBF paga minutos de kernel por su flexibilidad. La tabla final del notebook 07 pondrá estos números al lado de KNN (no linealidad local) y de los árboles.

**Siguiente notebook (05):** árbol de regresión — el modelo interpretable, con la demostración del sobreajuste y la poda por costo-complejidad.